# Customer Demographic Estimator

In [ ]:
from src.logic.customerAttributesEstimator import *
from IPython.display import Image, display, HTML
import os
from pathlib import Path
from typing import List

def get_all_paths(directory_path: str, recursive: bool = False) -> List[Path]:
    """
    Retrieves all file paths from a specified directory.
    
    Args:
        directory_path (str): The root folder to search.
        recursive (bool): If True, searches all subfolders. Defaults to False.
        
    Returns:
        List[Path]: A list of Path objects for every file found.
    """
    path_obj = Path(directory_path)
    
    # Check if the path actually exists to avoid errors
    if not path_obj.exists() or not path_obj.is_dir():
        return []

    if recursive:
        # rglob('*') finds all files and folders recursively
        return [p for p in path_obj.rglob('*') if p.is_file()]
    else:
        # iterdir() only looks at the immediate folder
        return [p for p in path_obj.iterdir() if p.is_file()]

def display_image_grid(image_paths, labels, cols=3):
    """
    Displays a grid of images with labels using HTML Flexbox.
    """
    html_content = '<div style="display: flex; flex-wrap: wrap; gap: 20px;">'
    
    for path, label in zip(image_paths, labels):
        # Create a container for each image + label
        item_html = f"""
        <div style="flex: 1 1 calc({100/cols}% - 20px); text-align: center; border: 1px solid #ddd; padding: 10px; border-radius: 8px;">
            <p style="font-weight: bold; margin-bottom: 8px;">{label}</p>
            <img src="{path}" style="width: 100%; height: auto; border-radius: 4px;">
        </div>
        """
        html_content += item_html
    
    html_content += '</div>'
    display(HTML(html_content))


# Usage
files = get_all_paths(os.path.join(os.getcwd(), "assets/faceDataPeople"), recursive=True)
image_files = []
image_labels = []
for file in files:
    estimation = estimateAttributes(file, 1)
    image_files.append(file)
    image_labels.append(f"SEX:{estimation[1]} AGE:{estimation[0]}")

display_image_grid(image_files, image_labels, cols=3)





# Simulate Customers

In [ ]:
from src.mockData.mock_data import *
simulate_customer(
    age="(15-20)",
    path_points=[(5, 5), (10, 15), (15, 25), (25, 30), (30, 40), (55, 10)],
    products=[chips, soda],
    sex="F",
)

# Display Customer Paths
This section uses `src/path_constructor.py` and Manim to animate every customer's path.
If Manim isn't installed, run `pip install manim` in your environment.


In [48]:
from pathlib import Path as FsPath
from collections import defaultdict
from src.database.database_setup import initialize_db, close_db, PathTable


def find_db_path(filename="store.db"):
    # Search upward from CWD for the database file.
    for candidate in [FsPath.cwd(), *FsPath.cwd().parents]:
        db_path = candidate / filename
        if db_path.exists():
            return db_path
    return None


def load_paths_from_db(db_path):
    # Load all customer paths directly from PathTable.
    initialize_db(str(db_path))
    try:
        rows = (
            PathTable
            .select(
                PathTable.customer_id,
                PathTable.timestamp,
                PathTable.location_x,
                PathTable.location_y,
                PathTable.path_id,
            )
            .order_by(
                PathTable.customer_id,
                PathTable.timestamp,
                PathTable.path_id,
            )
        )

        paths_by_customer = defaultdict(list)
        for row in rows:
            paths_by_customer[row.customer_id].append([row.location_x, row.location_y])

        return list(paths_by_customer.values())
    finally:
        close_db()


db_path = find_db_path()
if db_path is None:
    raise FileNotFoundError("store.db not found in current or parent directories")

all_paths = load_paths_from_db(db_path)
print(f"Loaded {len(all_paths)} customer paths from {db_path}")

# Keep paths in memory for inspection in the notebook.
all_paths[:3]


Loaded 6 customer paths from /Users/ryanpelchat/Documents/GitHub/project-group-12-0x5572414e657264/store.db


[[[10, 15], [15, 25], [25, 30], [30, 40], [55, 10], [5, 5]],
 [[30, 20], [45, 30], [50, 40], [55, 10], [25, 10]],
 [[15, 20], [45, 25], [50, 35], [55, 10], [10, 10]]]

In [49]:
%%manim -qm -v WARNING CustomerPathsScene
from pathlib import Path as FsPath
from collections import defaultdict
from manim import *
from src.database.database_setup import initialize_db, close_db, PathTable


def find_db_path(filename="store.db"):
    for candidate in [FsPath.cwd(), *FsPath.cwd().parents]:
        db_path = candidate / filename
        if db_path.exists():
            return db_path
    return None


class CustomerPathsScene(Scene):
    def construct(self):
        db_path = find_db_path()
        if db_path is None:
            self.add(Text("store.db not found").scale(0.7))
            self.wait(2)
            return

        initialize_db(str(db_path))
        try:
            rows = (
                PathTable
                .select(
                    PathTable.customer_id,
                    PathTable.timestamp,
                    PathTable.location_x,
                    PathTable.location_y,
                    PathTable.path_id,
                )
                .order_by(
                    PathTable.customer_id,
                    PathTable.timestamp,
                    PathTable.path_id,
                )
            )

            paths_by_customer = defaultdict(list)
            for row in rows:
                paths_by_customer[row.customer_id].append([row.location_x, row.location_y])

            paths = list(paths_by_customer.values())

            if not paths:
                self.add(Text("No customer paths found").scale(0.7))
                self.wait(2)
                return

            xs = [pt[0] for path in paths for pt in path]
            ys = [pt[1] for path in paths for pt in path]
            min_x, max_x = min(xs), max(xs)
            min_y, max_y = min(ys), max(ys)

            padding = 1
            axes = Axes(
                x_range=[min_x - padding, max_x + padding, 1],
                y_range=[min_y - padding, max_y + padding, 1],
                x_length=10,
                y_length=6,
                tips=False,
            )
            axes_labels = axes.get_axis_labels(x_label="X", y_label="Y")
            self.add(axes, axes_labels)

            colors = color_gradient(
                [BLUE, GREEN, YELLOW, ORANGE, RED, PURPLE],
                len(paths),
            )

            path_mobjects = []
            for path, color in zip(paths, colors):
                if len(path) == 1:
                    dot = Dot(axes.c2p(path[0][0], path[0][1]), color=color, radius=0.05)
                    path_mobjects.append(dot)
                    continue
                points = [axes.c2p(x, y) for x, y in path]
                vm = VMobject(color=color, stroke_width=2)
                vm.set_points_as_corners(points)
                path_mobjects.append(vm)

            self.play(
                LaggedStart(
                    *[Create(p) for p in path_mobjects],
                    lag_ratio=0.02,
                    run_time=6,
                )
            )
            self.wait(1)
        finally:
            close_db()


Manim Community v0.20.1

# Video People Tracking
This section uses `src/logic/videoTracker.py` (YOLOv8 + ByteTrack) to detect and track
people in a video, print the tracked coordinates, and generate an annotated output
video with colour-coded trails, IDs, and a HUD overlay.

In [ ]:
import os
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
from IPython.display import Video, display

from src.logic.videoTracker import (
    PersonTrack,
    TrackedPosition,
    track_people_in_video,
)

# ── Configuration ────────────────────────────────────────────────────────────
INPUT_VIDEO = os.path.join("test", "test_videos", "test_1.mp4")
OUTPUT_VIDEO = os.path.join("test", "test_videos", "test_1_tracked.mp4")

# Visually distinct BGR colours – wraps around for >10 people.
_PALETTE: List[Tuple[int, int, int]] = [
    (75, 25, 230),   # red
    (75, 180, 60),   # green
    (25, 225, 255),  # yellow
    (200, 130, 0),   # blue
    (48, 130, 245),  # orange
    (180, 30, 145),  # purple
    (240, 240, 70),  # cyan
    (230, 50, 240),  # magenta
    (60, 245, 210),  # lime
    (212, 190, 250), # pink
]

# ── Helper functions ─────────────────────────────────────────────────────────

def _colour(track_id: int) -> Tuple[int, int, int]:
    return _PALETTE[track_id % len(_PALETTE)]


def _video_offset_label(seconds: float) -> str:
    """Format *seconds* as ``MM:SS.mmm``."""
    mins = int(seconds // 60)
    secs = seconds % 60
    return f"{mins:02d}:{secs:06.3f}"


def _draw_text_with_background(
    frame: np.ndarray,
    text: str,
    origin: Tuple[int, int],
    colour: Tuple[int, int, int] = (255, 255, 255),
    scale: float = 0.5,
    thickness: int = 1,
) -> None:
    """Draw *text* with a dark rectangle behind it for readability."""
    font = cv2.FONT_HERSHEY_SIMPLEX
    (tw, th), baseline = cv2.getTextSize(text, font, scale, thickness)
    x, y = origin
    cv2.rectangle(
        frame,
        (x - 2, y - th - 4),
        (x + tw + 2, y + baseline + 2),
        (0, 0, 0),
        cv2.FILLED,
    )
    cv2.putText(frame, text, (x, y), font, scale, colour, thickness, cv2.LINE_AA)


def print_track_coordinates(
    tracks: List[PersonTrack],
    video_start: datetime,
    fps: float,
    total_frames: int,
    width: int,
    height: int,
    video_path: str,
) -> None:
    """Print a summary table of every tracked coordinate."""
    duration = total_frames / fps
    print("=" * 60)
    print("  VIDEO TRACKING RESULTS")
    print("=" * 60)
    print(f"  Source    : {video_path}")
    print(f"  FPS       : {fps:.1f}")
    print(f"  Frames    : {total_frames}")
    print(f"  Duration  : {duration:.1f}s")
    print(f"  Resolution: {width}x{height}")
    print(f"  People    : {len(tracks)}")
    print("=" * 60)

    if not tracks:
        print("\n  (no people detected)\n")
        return

    for track in tracks:
        first_off = (track.positions[0].timestamp - video_start).total_seconds()
        last_off = (track.positions[-1].timestamp - video_start).total_seconds()
        print(
            f"\n--- Person {track.track_id}  "
            f"({len(track.positions)} positions, "
            f"{_video_offset_label(first_off)} - "
            f"{_video_offset_label(last_off)}) ---"
        )
        for pos in track.positions:
            off = (pos.timestamp - video_start).total_seconds()
            print(f"  ({pos.x:4d}, {pos.y:4d})  @  {_video_offset_label(off)}")

    print()


def generate_tracking_video(
    tracks: List[PersonTrack],
    video_path: str,
    output_path: str,
    video_start: datetime,
) -> Optional[str]:
    """Re-read *video_path* and overlay tracked paths into *output_path*.

    Returns the final output path on success, or ``None`` on failure.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: cannot reopen video '{video_path}' for annotation.")
        return None

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    ok, first_frame = cap.read()
    if not ok:
        print("Error: video has no readable frames.")
        cap.release()
        return None
    real_h, real_w = first_frame.shape[:2]
    # H.264 requires even dimensions; round down if necessary.
    width = real_w & ~1
    height = real_h & ~1
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    out_dir = os.path.dirname(output_path)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)

    codec_candidates = [
        ("avc1", ".mp4"),
        ("mp4v", ".mp4"),
        ("XVID", ".avi"),
        ("MJPG", ".avi"),
    ]
    out = None
    for codec, ext in codec_candidates:
        candidate_path = os.path.splitext(output_path)[0] + ext
        fourcc = cv2.VideoWriter_fourcc(*codec)
        writer = cv2.VideoWriter(candidate_path, fourcc, fps, (width, height))
        if writer.isOpened():
            out = writer
            output_path = candidate_path
            print(f"  Using codec '{codec}' -> {output_path}")
            break
        writer.release()

    if out is None:
        print("Error: no working video codec found. Cannot write output.")
        cap.release()
        return None

    needs_resize = (real_w != width or real_h != height)

    # Pre-index: track_id -> [(frame_idx, x, y), ...]
    track_frames: Dict[int, List[Tuple[int, int, int]]] = {}
    for track in tracks:
        entries: List[Tuple[int, int, int]] = []
        for pos in track.positions:
            fi = round((pos.timestamp - video_start).total_seconds() * fps)
            entries.append((fi, pos.x, pos.y))
        track_frames[track.track_id] = entries

    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        for track_id, entries in track_frames.items():
            colour = _colour(track_id)

            trail = [(x, y) for fi, x, y in entries if fi <= frame_idx]
            if not trail:
                continue

            if len(trail) >= 2:
                pts = np.array(trail, dtype=np.int32).reshape(-1, 1, 2)
                cv2.polylines(frame, [pts], isClosed=False, color=colour, thickness=2)

            current = [(x, y) for fi, x, y in entries if fi == frame_idx]
            if current:
                cx, cy = current[-1]
                cv2.circle(frame, (cx, cy), 7, colour, cv2.FILLED)
                cv2.circle(frame, (cx, cy), 9, (255, 255, 255), 1)
                _draw_text_with_background(
                    frame,
                    f"ID:{track_id}",
                    (cx + 14, cy + 4),
                    colour=colour,
                    scale=0.5,
                    thickness=1,
                )

        offset_secs = frame_idx / fps
        _draw_text_with_background(
            frame,
            f"Frame {frame_idx}/{total_frames}  |  {_video_offset_label(offset_secs)}",
            (10, 25),
            scale=0.55,
            thickness=1,
        )
        _draw_text_with_background(
            frame,
            f"Tracked: {len(tracks)} people",
            (10, 50),
            scale=0.5,
            thickness=1,
        )

        if needs_resize:
            frame = cv2.resize(frame, (width, height))
        out.write(frame)
        frame_idx += 1

        if frame_idx % 100 == 0:
            pct = frame_idx / total_frames * 100 if total_frames else 0
            print(f"  rendering: frame {frame_idx}/{total_frames} ({pct:.0f}%)")

    cap.release()
    out.release()
    print(f"  Done – saved to: {output_path}")
    return output_path

print("Video tracking helpers loaded.")

In [ ]:
if not os.path.isfile(INPUT_VIDEO):
    raise FileNotFoundError(
        f"Test video not found at '{INPUT_VIDEO}'. "
        "Place your .mp4 file there and re-run this cell."
    )

probe = cv2.VideoCapture(INPUT_VIDEO)
fps = probe.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(probe.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(probe.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(probe.get(cv2.CAP_PROP_FRAME_HEIGHT))
probe.release()

video_start = datetime(2000, 1, 1)

print(f"Processing '{INPUT_VIDEO}' ...")
tracks = track_people_in_video(INPUT_VIDEO, video_start_time=video_start)
print(f"Tracking complete – {len(tracks)} people found.\n")

print_track_coordinates(tracks, video_start, fps, total_frames, width, height, INPUT_VIDEO)

print(f"Generating annotated video -> '{OUTPUT_VIDEO}' ...")
saved_path = generate_tracking_video(tracks, INPUT_VIDEO, OUTPUT_VIDEO, video_start)

if saved_path and os.path.isfile(saved_path):
    print("\nPlaying tracked video:")
    display(Video(saved_path, embed=True))

# Data Cleanup
Use the function below to delete all customer-related data from the database.


In [ ]:
from pathlib import Path as FsPath
from src.database.database_setup import (
    initialize_db,
    close_db,
    CustomerTable,
    PathTable,
    CheckoutTable,
    PurchaseTable,
    LogTable,
)


def delete_all_customer_data(db_path=None, include_logs=False):
    # Delete all customer-related data from the SQLite database.
    # This removes rows from: purchase, checkout, path, customer.
    # Set include_logs=True to also delete all log rows.
    if db_path is None:
        db_path = FsPath.cwd() / "store.db"

    db_path = FsPath(db_path)
    if not db_path.exists():
        raise FileNotFoundError(f"Database not found: {db_path}")

    initialize_db(str(db_path))
    try:
        deleted = {}
        deleted["purchase"] = PurchaseTable.delete().execute()
        deleted["checkout"] = CheckoutTable.delete().execute()
        deleted["path"] = PathTable.delete().execute()
        deleted["customer"] = CustomerTable.delete().execute()
        if include_logs:
            deleted["log"] = LogTable.delete().execute()
        return deleted
    finally:
        close_db()

delete_all_customer_data(include_logs=False)
